# ✈️ Global Airline Stock Valuation Dashboard
## Comprehensive Financial & Operational Analysis

**Objective:** Identify undervalued airline stocks globally through fundamental analysis, financial ratio screening, airline-specific operational metrics, and composite valuation scoring.

**Coverage:** 25+ airlines across North America, Europe, Asia-Pacific, and Latin America

**Methodology:**
- Financial data pulled via `yfinance` (Yahoo Finance API)
- Airline-specific capacity/profitability metrics derived from financial statements
- Composite valuation score ranking airlines on 12+ factors
- Interactive Plotly visualisations for every dimension of analysis

**Key Metrics Analysed:**
| Category | Metrics |
|----------|---------|
| **Valuation** | P/E, Forward P/E, P/B, P/S, EV/EBITDA, EV/Revenue |
| **Profitability** | Gross Margin, Operating Margin, Net Margin, ROE, ROA, ROIC |
| **Leverage** | Debt/Equity, Net Debt/EBITDA, Interest Coverage, Current Ratio |
| **Efficiency** | Revenue/Employee, CAPEX/Revenue, Asset Turnover |
| **Airline-Specific** | Revenue per ASK (proxy), EBITDAR Margin, Fleet CAPEX Intensity, FCF Yield |
| **Growth** | Revenue Growth, Earnings Growth, FCF Growth |
| **Momentum** | 52-Week Performance, Distance from 52-Week High, Beta |

---
*Disclaimer: This is for educational and analytical purposes only. Not financial advice. Always conduct your own due diligence.*


In [1]:
# ============================================================
# SETUP: Install packages (uncomment for Kaggle/Colab)
# ============================================================
!pip install yfinance plotly kaleido scipy pandas numpy -q

import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
from datetime import datetime, timedelta
from scipy import stats
import time
import json

# Plotly settings
pio.templates.default = "plotly_dark"
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

print("✅ All packages loaded successfully")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.2 MB/s eta 0:00:00
✅ All packages loaded successfully
📅 Analysis Date: 2026-02-23 17:36


## 1. Airline Universe Definition

We cover airlines across all major global markets, segmented by:
- **Region**: North America, Europe, Asia-Pacific, Latin America
- **Type**: Full-Service Carrier (FSC), Low-Cost Carrier (LCC), Ultra Low-Cost (ULCC)


In [2]:
# ============================================================
# AIRLINE UNIVERSE - Global Coverage
# ============================================================

AIRLINES = {
    # --- NORTH AMERICA ---
    'DAL':    {'name': 'Delta Air Lines',     'region': 'North America', 'type': 'FSC',  'currency': 'USD', 'hub': 'Atlanta'},
    'UAL':    {'name': 'United Airlines',      'region': 'North America', 'type': 'FSC',  'currency': 'USD', 'hub': 'Chicago/Newark'},
    'AAL':    {'name': 'American Airlines',    'region': 'North America', 'type': 'FSC',  'currency': 'USD', 'hub': 'Dallas/Miami'},
    'LUV':    {'name': 'Southwest Airlines',   'region': 'North America', 'type': 'LCC',  'currency': 'USD', 'hub': 'Dallas Love'},
    'JBLU':   {'name': 'JetBlue Airways',      'region': 'North America', 'type': 'LCC',  'currency': 'USD', 'hub': 'New York JFK'},
    'ALK':    {'name': 'Alaska Air Group',     'region': 'North America', 'type': 'FSC',  'currency': 'USD', 'hub': 'Seattle'},
    'ALGT':   {'name': 'Allegiant Travel',     'region': 'North America', 'type': 'ULCC', 'currency': 'USD', 'hub': 'Las Vegas'},
    'SKYW':   {'name': 'SkyWest Inc',          'region': 'North America', 'type': 'Regional', 'currency': 'USD', 'hub': 'St. George'},
    'AC.TO':  {'name': 'Air Canada',           'region': 'North America', 'type': 'FSC',  'currency': 'CAD', 'hub': 'Toronto/Montreal'},
    'CPA':    {'name': 'Copa Holdings',        'region': 'Latin America', 'type': 'FSC',  'currency': 'USD', 'hub': 'Panama City'},
    
    # --- EUROPE ---
    'IAG.L':  {'name': 'IAG (BA/Iberia)',      'region': 'Europe',        'type': 'FSC',  'currency': 'GBP', 'hub': 'London/Madrid'},
    'AF.PA':  {'name': 'Air France-KLM',       'region': 'Europe',        'type': 'FSC',  'currency': 'EUR', 'hub': 'Paris/Amsterdam'},
    'LHA.DE': {'name': 'Lufthansa Group',      'region': 'Europe',        'type': 'FSC',  'currency': 'EUR', 'hub': 'Frankfurt/Munich'},
    'RYAAY':  {'name': 'Ryanair Holdings',     'region': 'Europe',        'type': 'ULCC', 'currency': 'USD', 'hub': 'Dublin'},
    'EZJ.L':  {'name': 'easyJet',              'region': 'Europe',        'type': 'LCC',  'currency': 'GBP', 'hub': 'London Luton'},
    'WIZZ.L': {'name': 'Wizz Air',             'region': 'Europe',        'type': 'ULCC', 'currency': 'GBP', 'hub': 'Budapest'},
    'TKC':    {'name': 'Turkish Airlines',     'region': 'Europe/ME',     'type': 'FSC',  'currency': 'USD', 'hub': 'Istanbul'},
    
    # --- ASIA-PACIFIC ---
    '9201.T': {'name': 'Japan Airlines',       'region': 'Asia-Pacific',  'type': 'FSC',  'currency': 'JPY', 'hub': 'Tokyo Haneda'},
    '9202.T': {'name': 'ANA Holdings',         'region': 'Asia-Pacific',  'type': 'FSC',  'currency': 'JPY', 'hub': 'Tokyo Narita'},
    'QAN.AX': {'name': 'Qantas Airways',       'region': 'Asia-Pacific',  'type': 'FSC',  'currency': 'AUD', 'hub': 'Sydney'},
    'C6L.SI': {'name': 'Singapore Airlines',   'region': 'Asia-Pacific',  'type': 'FSC',  'currency': 'SGD', 'hub': 'Singapore'},
    '0293.HK':{'name': 'Cathay Pacific',       'region': 'Asia-Pacific',  'type': 'FSC',  'currency': 'HKD', 'hub': 'Hong Kong'},
    'AIR.NZ': {'name': 'Air New Zealand',      'region': 'Asia-Pacific',  'type': 'FSC',  'currency': 'NZD', 'hub': 'Auckland'},
    'INDIGO.NS':{'name': 'IndiGo (InterGlobe)','region': 'Asia-Pacific',  'type': 'LCC',  'currency': 'INR', 'hub': 'Delhi'},
}

print(f"📊 Universe: {len(AIRLINES)} airlines across {len(set(a['region'] for a in AIRLINES.values()))} regions")
print(f"   FSC: {sum(1 for a in AIRLINES.values() if a['type']=='FSC')}")
print(f"   LCC: {sum(1 for a in AIRLINES.values() if a['type']=='LCC')}")
print(f"   ULCC: {sum(1 for a in AIRLINES.values() if a['type']=='ULCC')}")
print(f"   Regional: {sum(1 for a in AIRLINES.values() if a['type']=='Regional')}")


📊 Universe: 24 airlines across 5 regions
   FSC: 16
   LCC: 4
   ULCC: 3
   Regional: 1


## 2. Data Collection Engine

Pull comprehensive financial data for every airline including:
- Company info & current market data
- Income statements (4 years)
- Balance sheets (4 years)
- Cash flow statements (4 years)
- Historical price data (1 year)


In [3]:
# ============================================================
# DATA COLLECTION - Pull all financial data
# ============================================================

def fetch_airline_data(ticker, meta, max_retries=3):
    """Fetch comprehensive financial data for a single airline."""
    for attempt in range(max_retries):
        try:
            t = yf.Ticker(ticker)
            info = t.info
            
            # Validate we got real data
            mc = info.get('marketCap', 0)
            if not mc or mc == 0:
                return None
            
            # Pull financial statements
            income = t.income_stmt
            balance = t.balance_sheet
            cashflow = t.cash_flow
            
            # 1-year price history
            hist = t.history(period='1y')
            
            return {
                'ticker': ticker,
                'meta': meta,
                'info': info,
                'income': income,
                'balance': balance,
                'cashflow': cashflow,
                'history': hist
            }
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                print(f"  ⚠️ Failed: {ticker} ({meta['name']}): {e}")
                return None

# Fetch all airlines
print("🔄 Fetching financial data for all airlines...")
print("=" * 60)

airline_data = {}
failed = []

for i, (ticker, meta) in enumerate(AIRLINES.items()):
    print(f"  [{i+1}/{len(AIRLINES)}] {ticker:12s} {meta['name']:30s}", end=" ")
    data = fetch_airline_data(ticker, meta)
    if data:
        airline_data[ticker] = data
        mc = data['info'].get('marketCap', 0)
        print(f"✅ MCap: ${mc/1e9:.1f}B")
    else:
        failed.append(ticker)
        print("❌ FAILED")
    time.sleep(0.5)  # Rate limiting

print(f"\n{'='*60}")
print(f"✅ Successfully loaded: {len(airline_data)}/{len(AIRLINES)} airlines")
if failed:
    print(f"❌ Failed: {', '.join(failed)}")


🔄 Fetching financial data for all airlines...
  [1/24] DAL          Delta Air Lines                ✅ MCap: $43.7B
  [2/24] UAL          United Airlines                ✅ MCap: $34.9B
  [3/24] AAL          American Airlines              ✅ MCap: $8.5B
  [4/24] LUV          Southwest Airlines             ✅ MCap: $26.0B
  [5/24] JBLU         JetBlue Airways                ✅ MCap: $2.2B
  [6/24] ALK          Alaska Air Group               ✅ MCap: $5.8B
  [7/24] ALGT         Allegiant Travel               ✅ MCap: $1.9B
  [8/24] SKYW         SkyWest Inc                    ✅ MCap: $4.1B
  [9/24] AC.TO        Air Canada                     ✅ MCap: $6.0B
  [10/24] CPA          Copa Holdings                  ✅ MCap: $5.8B
  [11/24] IAG.L        IAG (BA/Iberia)                ✅ MCap: $19.6B
  [12/24] AF.PA        Air France-KLM                 ✅ MCap: $3.5B
  [13/24] LHA.DE       Lufthansa Group                ✅ MCap: $11.0B
  [14/24] RYAAY        Ryanair Holdings               ✅ MCap: $33.8B
  [15

## 3. Financial Metrics Extraction

Extract and calculate 40+ metrics across valuation, profitability, leverage, efficiency, and growth dimensions.


In [4]:
# ============================================================
# METRICS EXTRACTION - Build comprehensive metrics DataFrame
# ============================================================

def safe_get(d, key, default=np.nan):
    """Safely extract a value from dict."""
    val = d.get(key, default)
    if val is None:
        return default
    return val

def safe_stmt_get(stmt, row_name, col_idx=0):
    """Safely extract a value from a financial statement DataFrame."""
    try:
        if stmt is None or stmt.empty:
            return np.nan
        if row_name in stmt.index:
            val = stmt.loc[row_name].iloc[col_idx]
            if pd.notna(val):
                return float(val)
        return np.nan
    except:
        return np.nan

def compute_growth(stmt, row_name, periods=1):
    """Compute YoY growth rate from financial statement."""
    try:
        if stmt is None or stmt.empty or row_name not in stmt.index:
            return np.nan
        row = stmt.loc[row_name].dropna()
        if len(row) < periods + 1:
            return np.nan
        current = float(row.iloc[0])
        prior = float(row.iloc[periods])
        if prior == 0 or prior < 0:
            return np.nan
        return (current - prior) / abs(prior)
    except:
        return np.nan

def extract_metrics(ticker, data):
    """Extract comprehensive metrics for a single airline."""
    info = data['info']
    inc = data['income']
    bs = data['balance']
    cf = data['cashflow']
    hist = data['history']
    meta = data['meta']
    
    m = {}
    
    # --- IDENTIFIERS ---
    m['Ticker'] = ticker
    m['Name'] = meta['name']
    m['Region'] = meta['region']
    m['Type'] = meta['type']
    m['Currency'] = meta['currency']
    
    # --- MARKET DATA ---
    m['Market Cap ($B)'] = safe_get(info, 'marketCap', 0) / 1e9
    m['Enterprise Value ($B)'] = safe_get(info, 'enterpriseValue', 0) / 1e9
    m['Current Price'] = safe_get(info, 'currentPrice')
    m['52W High'] = safe_get(info, 'fiftyTwoWeekHigh')
    m['52W Low'] = safe_get(info, 'fiftyTwoWeekLow')
    m['Beta'] = safe_get(info, 'beta')
    
    # Distance from 52W high (%)
    if pd.notna(m['Current Price']) and pd.notna(m['52W High']) and m['52W High'] > 0:
        m['Dist from 52W High (%)'] = ((m['Current Price'] - m['52W High']) / m['52W High']) * 100
    else:
        m['Dist from 52W High (%)'] = np.nan
    
    # --- VALUATION RATIOS ---
    m['Trailing P/E'] = safe_get(info, 'trailingPE')
    m['Forward P/E'] = safe_get(info, 'forwardPE')
    m['P/B'] = safe_get(info, 'priceToBook')
    m['P/S'] = safe_get(info, 'priceToSalesTrailing12Months')
    m['EV/EBITDA'] = safe_get(info, 'enterpriseToEbitda')
    m['EV/Revenue'] = safe_get(info, 'enterpriseToRevenue')
    
    # --- PROFITABILITY ---
    m['Gross Margin (%)'] = safe_get(info, 'grossMargins', np.nan) 
    if pd.notna(m['Gross Margin (%)']):
        m['Gross Margin (%)'] *= 100
    
    m['Operating Margin (%)'] = safe_get(info, 'operatingMargins', np.nan)
    if pd.notna(m['Operating Margin (%)']):
        m['Operating Margin (%)'] *= 100
    
    m['Net Margin (%)'] = safe_get(info, 'profitMargins', np.nan)
    if pd.notna(m['Net Margin (%)']):
        m['Net Margin (%)'] *= 100
    
    m['ROE (%)'] = safe_get(info, 'returnOnEquity', np.nan)
    if pd.notna(m['ROE (%)']):
        m['ROE (%)'] *= 100
    
    m['ROA (%)'] = safe_get(info, 'returnOnAssets', np.nan)
    if pd.notna(m['ROA (%)']):
        m['ROA (%)'] *= 100
    
    # EBITDA Margin from statements
    revenue_ttm = safe_get(info, 'totalRevenue', np.nan)
    ebitda_ttm = safe_get(info, 'ebitda', np.nan)
    if pd.notna(revenue_ttm) and pd.notna(ebitda_ttm) and revenue_ttm > 0:
        m['EBITDA Margin (%)'] = (ebitda_ttm / revenue_ttm) * 100
    else:
        m['EBITDA Margin (%)'] = np.nan
    
    # EBITDAR Margin (EBITDA + Aircraft Rent - key airline metric)
    # Approximate: EBITDA + Operating Lease expenses
    ebitda_val = safe_stmt_get(inc, 'EBITDA')
    rev_val = safe_stmt_get(inc, 'Total Revenue')
    if pd.notna(ebitda_val) and pd.notna(rev_val) and rev_val > 0:
        # Try to find lease costs from operating expenses
        lease_cost = safe_stmt_get(inc, 'Operating Lease Cost')
        if pd.isna(lease_cost):
            lease_cost = 0
        m['EBITDAR Margin (%)'] = ((ebitda_val + lease_cost) / rev_val) * 100
    else:
        m['EBITDAR Margin (%)'] = m.get('EBITDA Margin (%)', np.nan)
    
    # --- ROIC Calculation ---
    nopat = safe_stmt_get(inc, 'EBIT')
    tax_rate = safe_stmt_get(inc, 'Tax Rate For Calcs')
    if pd.isna(tax_rate):
        tax_rate = 0.25  # Assume 25%
    total_equity = safe_stmt_get(bs, 'Stockholders Equity')
    total_debt = safe_stmt_get(bs, 'Total Debt')
    cash = safe_stmt_get(bs, 'Cash And Cash Equivalents')
    if pd.notna(nopat) and pd.notna(total_equity) and pd.notna(total_debt):
        invested_capital = total_equity + total_debt - (cash if pd.notna(cash) else 0)
        if invested_capital > 0:
            m['ROIC (%)'] = (nopat * (1 - tax_rate) / invested_capital) * 100
        else:
            m['ROIC (%)'] = np.nan
    else:
        m['ROIC (%)'] = np.nan
    
    # --- LEVERAGE & LIQUIDITY ---
    m['Debt/Equity'] = safe_get(info, 'debtToEquity', np.nan)
    if pd.notna(m['Debt/Equity']):
        m['Debt/Equity'] /= 100  # yfinance returns as percentage
    
    m['Current Ratio'] = safe_get(info, 'currentRatio')
    m['Quick Ratio'] = safe_get(info, 'quickRatio')
    
    # Net Debt / EBITDA
    net_debt = safe_get(info, 'enterpriseValue', 0) - safe_get(info, 'marketCap', 0)
    if pd.notna(ebitda_ttm) and ebitda_ttm > 0:
        m['Net Debt/EBITDA'] = net_debt / ebitda_ttm
    else:
        m['Net Debt/EBITDA'] = np.nan
    
    # Interest Coverage
    ebit = safe_stmt_get(inc, 'EBIT')
    interest = safe_stmt_get(inc, 'Interest Expense')
    if pd.notna(ebit) and pd.notna(interest) and abs(interest) > 0:
        m['Interest Coverage'] = ebit / abs(interest)
    else:
        m['Interest Coverage'] = np.nan
    
    # --- CASH FLOW ---
    m['Operating CF ($B)'] = safe_get(info, 'operatingCashflow', 0) / 1e9
    m['Free Cash Flow ($B)'] = safe_get(info, 'freeCashflow', 0) / 1e9
    
    # FCF Yield
    mcap = safe_get(info, 'marketCap', 0)
    fcf = safe_get(info, 'freeCashflow', 0)
    if mcap > 0 and fcf != 0:
        m['FCF Yield (%)'] = (fcf / mcap) * 100
    else:
        m['FCF Yield (%)'] = np.nan
    
    # FCF Margin
    if pd.notna(revenue_ttm) and revenue_ttm > 0 and fcf != 0:
        m['FCF Margin (%)'] = (fcf / revenue_ttm) * 100
    else:
        m['FCF Margin (%)'] = np.nan
    
    # --- EFFICIENCY & AIRLINE-SPECIFIC ---
    employees = safe_get(info, 'fullTimeEmployees', np.nan)
    if pd.notna(employees) and employees > 0 and pd.notna(revenue_ttm):
        m['Revenue/Employee ($K)'] = (revenue_ttm / employees) / 1000
    else:
        m['Revenue/Employee ($K)'] = np.nan
    
    # CAPEX Intensity (CAPEX / Revenue)
    capex = safe_stmt_get(cf, 'Capital Expenditure')
    if pd.notna(capex) and pd.notna(rev_val) and rev_val > 0:
        m['CAPEX/Revenue (%)'] = (abs(capex) / rev_val) * 100
    else:
        m['CAPEX/Revenue (%)'] = np.nan
    
    # Asset Turnover
    total_assets = safe_stmt_get(bs, 'Total Assets')
    if pd.notna(rev_val) and pd.notna(total_assets) and total_assets > 0:
        m['Asset Turnover'] = rev_val / total_assets
    else:
        m['Asset Turnover'] = np.nan
    
    # --- GROWTH ---
    m['Revenue Growth (%)'] = safe_get(info, 'revenueGrowth', np.nan)
    if pd.notna(m['Revenue Growth (%)']):
        m['Revenue Growth (%)'] *= 100
    
    m['Earnings Growth (%)'] = safe_get(info, 'earningsGrowth', np.nan)
    if pd.notna(m['Earnings Growth (%)']):
        m['Earnings Growth (%)'] *= 100
    
    # Revenue CAGR (3-year) from income statement
    rev_growth_3y = compute_growth(inc, 'Total Revenue', periods=3)
    if pd.notna(rev_growth_3y):
        m['Rev CAGR 3Y (%)'] = ((1 + rev_growth_3y) ** (1/3) - 1) * 100
    else:
        m['Rev CAGR 3Y (%)'] = np.nan
    
    # --- DIVIDEND ---
    m['Dividend Yield (%)'] = safe_get(info, 'dividendYield', 0)
    if pd.notna(m['Dividend Yield (%)']) and m['Dividend Yield (%)'] > 0:
        m['Dividend Yield (%)'] *= 100
    
    m['Payout Ratio (%)'] = safe_get(info, 'payoutRatio', 0)
    if pd.notna(m['Payout Ratio (%)']) and m['Payout Ratio (%)'] > 0:
        m['Payout Ratio (%)'] *= 100
    
    # --- ANALYST TARGETS ---
    m['Analyst Target ($)'] = safe_get(info, 'targetMeanPrice')
    m['Analyst Recommendation'] = safe_get(info, 'recommendationKey', 'N/A')
    m['# Analysts'] = safe_get(info, 'numberOfAnalystOpinions', 0)
    
    # Upside to analyst target
    if pd.notna(m['Analyst Target ($)']) and pd.notna(m['Current Price']) and m['Current Price'] > 0:
        m['Analyst Upside (%)'] = ((m['Analyst Target ($)'] - m['Current Price']) / m['Current Price']) * 100
    else:
        m['Analyst Upside (%)'] = np.nan
    
    # --- PRICE PERFORMANCE ---
    if hist is not None and len(hist) > 0:
        prices = hist['Close']
        if len(prices) >= 252:
            m['1Y Return (%)'] = ((prices.iloc[-1] / prices.iloc[0]) - 1) * 100
        elif len(prices) > 20:
            m['1Y Return (%)'] = ((prices.iloc[-1] / prices.iloc[0]) - 1) * 100
        else:
            m['1Y Return (%)'] = np.nan
        
        if len(prices) >= 126:
            m['6M Return (%)'] = ((prices.iloc[-1] / prices.iloc[-126]) - 1) * 100
        else:
            m['6M Return (%)'] = np.nan
        
        if len(prices) >= 63:
            m['3M Return (%)'] = ((prices.iloc[-1] / prices.iloc[-63]) - 1) * 100
        else:
            m['3M Return (%)'] = np.nan
        
        if len(prices) >= 21:
            m['1M Return (%)'] = ((prices.iloc[-1] / prices.iloc[-21]) - 1) * 100
        else:
            m['1M Return (%)'] = np.nan
        
        # Volatility (annualised)
        returns = prices.pct_change().dropna()
        if len(returns) > 20:
            m['Annualised Vol (%)'] = returns.std() * np.sqrt(252) * 100
        else:
            m['Annualised Vol (%)'] = np.nan
    else:
        m['1Y Return (%)'] = np.nan
        m['6M Return (%)'] = np.nan
        m['3M Return (%)'] = np.nan
        m['1M Return (%)'] = np.nan
        m['Annualised Vol (%)'] = np.nan
    
    # Employees (for reference)
    m['Employees'] = employees
    
    # Revenue ($B)
    m['Revenue ($B)'] = revenue_ttm / 1e9 if pd.notna(revenue_ttm) else np.nan
    
    return m

# Extract metrics for all airlines
print("📊 Extracting comprehensive metrics...")
metrics_list = []

for ticker, data in airline_data.items():
    metrics = extract_metrics(ticker, data)
    metrics_list.append(metrics)

df = pd.DataFrame(metrics_list)
df = df.set_index('Ticker')

# Filter out airlines with P/E > 200 or < 0 for cleaner analysis
df.loc[df['Trailing P/E'] > 200, 'Trailing P/E'] = np.nan
df.loc[df['Trailing P/E'] < 0, 'Trailing P/E'] = np.nan

print(f"\n✅ Extracted {len(df.columns)} metrics for {len(df)} airlines")
print(f"\n📋 Metrics summary:")
print(df[['Name', 'Region', 'Type', 'Market Cap ($B)', 'Trailing P/E', 'EV/EBITDA', 'Net Margin (%)', 'ROE (%)']].to_string())


📊 Extracting comprehensive metrics...

✅ Extracted 53 metrics for 24 airlines

📋 Metrics summary:
                          Name         Region      Type  Market Cap ($B)  Trailing P/E  EV/EBITDA  Net Margin (%)  ROE (%)
Ticker                                                                                                                    
DAL            Delta Air Lines  North America       FSC            43.70          8.74       8.00            7.90    27.69
UAL            United Airlines  North America       FSC            34.90         10.57       7.14            5.68    23.99
AAL          American Airlines  North America       FSC             8.53         75.97      10.09            0.20      NaN
LUV         Southwest Airlines  North America       LCC            26.04         63.73      16.48            1.57     4.81
JBLU           JetBlue Airways  North America       LCC             2.15           NaN      33.34           -6.64   -25.29
ALK           Alaska Air Group  North Ame

## 4. Valuation Analysis

### 4.1 Valuation Multiples Comparison
The most important airline valuation metrics: **P/E, EV/EBITDA, P/B, and EV/Revenue**. Lower multiples relative to profitability indicate potential undervaluation.


In [5]:
# ============================================================
# VALUATION MULTIPLES - Comprehensive Comparison
# ============================================================

# Colour scheme
COLORS = {
    'North America': '#00D4AA',
    'Europe': '#FF6B6B',
    'Asia-Pacific': '#4ECDC4',
    'Latin America': '#FFE66D',
    'Europe/ME': '#FF8C42',
}

TYPE_COLORS = {
    'FSC': '#636EFA',
    'LCC': '#00CC96',
    'ULCC': '#EF553B',
    'Regional': '#AB63FA',
}

def region_color(name):
    return COLORS.get(name, '#888888')

# --- P/E Comparison ---
val_df = df[['Name', 'Region', 'Type', 'Trailing P/E', 'Forward P/E', 'P/B', 'P/S', 
             'EV/EBITDA', 'EV/Revenue', 'Market Cap ($B)']].dropna(subset=['Trailing P/E']).copy()
val_df = val_df.sort_values('Trailing P/E')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Trailing P/E Ratio', 'EV/EBITDA', 'Price/Book (P/B)', 'EV/Revenue'),
    vertical_spacing=0.12, horizontal_spacing=0.08
)

# P/E
for region in val_df['Region'].unique():
    mask = val_df['Region'] == region
    subset = val_df[mask].sort_values('Trailing P/E')
    fig.add_trace(go.Bar(
        x=subset['Name'], y=subset['Trailing P/E'],
        name=region, marker_color=region_color(region),
        text=subset['Trailing P/E'].round(1), textposition='outside',
        showlegend=True, legendgroup=region
    ), row=1, col=1)

# EV/EBITDA
ev_df = df[['Name', 'Region', 'EV/EBITDA']].dropna(subset=['EV/EBITDA']).copy()
ev_df = ev_df[ev_df['EV/EBITDA'] > 0].sort_values('EV/EBITDA')
for region in ev_df['Region'].unique():
    mask = ev_df['Region'] == region
    subset = ev_df[mask]
    fig.add_trace(go.Bar(
        x=subset['Name'], y=subset['EV/EBITDA'],
        marker_color=region_color(region),
        text=subset['EV/EBITDA'].round(1), textposition='outside',
        showlegend=False, legendgroup=region
    ), row=1, col=2)

# P/B
pb_df = df[['Name', 'Region', 'P/B']].dropna(subset=['P/B']).copy()
pb_df = pb_df[pb_df['P/B'] > 0].sort_values('P/B')
for region in pb_df['Region'].unique():
    mask = pb_df['Region'] == region
    subset = pb_df[mask]
    fig.add_trace(go.Bar(
        x=subset['Name'], y=subset['P/B'],
        marker_color=region_color(region),
        text=subset['P/B'].round(2), textposition='outside',
        showlegend=False, legendgroup=region
    ), row=2, col=1)

# EV/Revenue
evr_df = df[['Name', 'Region', 'EV/Revenue']].dropna(subset=['EV/Revenue']).copy()
evr_df = evr_df[evr_df['EV/Revenue'] > 0].sort_values('EV/Revenue')
for region in evr_df['Region'].unique():
    mask = evr_df['Region'] == region
    subset = evr_df[mask]
    fig.add_trace(go.Bar(
        x=subset['Name'], y=subset['EV/Revenue'],
        marker_color=region_color(region),
        text=subset['EV/Revenue'].round(2), textposition='outside',
        showlegend=False, legendgroup=region
    ), row=2, col=2)

fig.update_layout(
    height=900, width=1400,
    title_text="<b>Global Airline Valuation Multiples</b>",
    title_font_size=22,
    barmode='group',
    font=dict(size=10),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.update_xaxes(tickangle=45, tickfont_size=8)
fig.show()


### 4.2 Profitability Analysis
Operating margins and EBITDA margins are critical for airlines. EBITDAR margin (EBITDA + rent/lease costs) is the airline industry standard as it neutralises fleet ownership structure differences.


In [6]:
# ============================================================
# PROFITABILITY ANALYSIS
# ============================================================

prof_cols = ['Name', 'Region', 'Type', 'Gross Margin (%)', 'Operating Margin (%)', 
             'Net Margin (%)', 'EBITDA Margin (%)', 'EBITDAR Margin (%)', 'ROE (%)', 'ROA (%)', 'ROIC (%)']
prof_df = df[prof_cols].copy()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Operating Margin (%)', 'EBITDA Margin (%)',
        'Return on Equity (%)', 'Return on Invested Capital (%)'
    ),
    vertical_spacing=0.15, horizontal_spacing=0.08
)

# Operating Margin
om_df = prof_df.dropna(subset=['Operating Margin (%)']).sort_values('Operating Margin (%)', ascending=True)
colors_om = [region_color(r) for r in om_df['Region']]
fig.add_trace(go.Bar(
    y=om_df['Name'], x=om_df['Operating Margin (%)'],
    orientation='h', marker_color=colors_om,
    text=om_df['Operating Margin (%)'].round(1), textposition='outside',
    showlegend=False
), row=1, col=1)

# EBITDA Margin
em_df = prof_df.dropna(subset=['EBITDA Margin (%)']).sort_values('EBITDA Margin (%)', ascending=True)
colors_em = [region_color(r) for r in em_df['Region']]
fig.add_trace(go.Bar(
    y=em_df['Name'], x=em_df['EBITDA Margin (%)'],
    orientation='h', marker_color=colors_em,
    text=em_df['EBITDA Margin (%)'].round(1), textposition='outside',
    showlegend=False
), row=1, col=2)

# ROE
roe_df = prof_df.dropna(subset=['ROE (%)']).sort_values('ROE (%)', ascending=True)
colors_roe = [region_color(r) for r in roe_df['Region']]
fig.add_trace(go.Bar(
    y=roe_df['Name'], x=roe_df['ROE (%)'],
    orientation='h', marker_color=colors_roe,
    text=roe_df['ROE (%)'].round(1), textposition='outside',
    showlegend=False
), row=2, col=1)

# ROIC
roic_df = prof_df.dropna(subset=['ROIC (%)']).sort_values('ROIC (%)', ascending=True)
colors_roic = [region_color(r) for r in roic_df['Region']]
fig.add_trace(go.Bar(
    y=roic_df['Name'], x=roic_df['ROIC (%)'],
    orientation='h', marker_color=colors_roic,
    text=roic_df['ROIC (%)'].round(1), textposition='outside',
    showlegend=False
), row=2, col=2)

fig.update_layout(
    height=1000, width=1400,
    title_text="<b>Profitability Analysis - Global Airlines</b>",
    title_font_size=22,
    font=dict(size=9),
)
fig.show()

# Print profitability table
print("\n📊 PROFITABILITY METRICS TABLE:")
print("=" * 120)
display_cols = ['Name', 'Type', 'Operating Margin (%)', 'EBITDA Margin (%)', 'Net Margin (%)', 'ROE (%)', 'ROIC (%)']
print(prof_df[display_cols].sort_values('Operating Margin (%)', ascending=False).to_string(index=True))



📊 PROFITABILITY METRICS TABLE:
                          Name      Type  Operating Margin (%)  EBITDA Margin (%)  Net Margin (%)  ROE (%)  ROIC (%)
Ticker                                                                                                              
IAG.L          IAG (BA/Iberia)       FSC                 22.26              18.86            9.30      NaN     22.35
CPA              Copa Holdings       FSC                 21.77              32.73           18.57    26.09     18.12
EZJ.L                  easyJet       LCC                 16.31               9.95            4.89    15.27     12.44
INDIGO.NS  IndiGo (InterGlobe)       LCC                 15.76              13.44            3.79      NaN     16.13
SKYW               SkyWest Inc  Regional                 13.10              24.21           10.56    16.62       NaN
ALGT          Allegiant Travel      ULCC                 12.91              16.47           -1.71    -4.25     -5.21
9201.T          Japan Airlines  

### 4.3 Leverage & Financial Health

Airlines are capital-intensive with high debt loads from fleet financing. Key metrics:
- **Net Debt/EBITDA** < 3x is healthy; > 5x is concerning
- **Interest Coverage** > 3x means comfortable debt servicing
- **Current Ratio** < 1.0 is normal for airlines (advance ticket sales create deferred revenue)


In [7]:
# ============================================================
# LEVERAGE & FINANCIAL HEALTH
# ============================================================

lev_cols = ['Name', 'Region', 'Type', 'Debt/Equity', 'Net Debt/EBITDA', 
            'Interest Coverage', 'Current Ratio', 'Quick Ratio']
lev_df = df[lev_cols].copy()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Net Debt / EBITDA (Lower = Healthier)', 'Debt / Equity Ratio',
        'Interest Coverage Ratio (Higher = Safer)', 'Current Ratio'
    ),
    vertical_spacing=0.15, horizontal_spacing=0.1
)

# Net Debt/EBITDA
nd_df = lev_df.dropna(subset=['Net Debt/EBITDA']).sort_values('Net Debt/EBITDA', ascending=True)
nd_df_clean = nd_df[(nd_df['Net Debt/EBITDA'] > -5) & (nd_df['Net Debt/EBITDA'] < 15)]
colors = ['#FF4444' if x > 5 else '#FFAA00' if x > 3 else '#00CC66' for x in nd_df_clean['Net Debt/EBITDA']]
fig.add_trace(go.Bar(
    y=nd_df_clean['Name'], x=nd_df_clean['Net Debt/EBITDA'],
    orientation='h', marker_color=colors,
    text=nd_df_clean['Net Debt/EBITDA'].round(1), textposition='outside',
    showlegend=False
), row=1, col=1)

# Debt/Equity
de_df = lev_df.dropna(subset=['Debt/Equity']).sort_values('Debt/Equity', ascending=True)
de_df_clean = de_df[(de_df['Debt/Equity'] > -2) & (de_df['Debt/Equity'] < 10)]
colors = ['#FF4444' if x > 3 else '#FFAA00' if x > 1.5 else '#00CC66' for x in de_df_clean['Debt/Equity']]
fig.add_trace(go.Bar(
    y=de_df_clean['Name'], x=de_df_clean['Debt/Equity'],
    orientation='h', marker_color=colors,
    text=de_df_clean['Debt/Equity'].round(2), textposition='outside',
    showlegend=False
), row=1, col=2)

# Interest Coverage
ic_df = lev_df.dropna(subset=['Interest Coverage']).sort_values('Interest Coverage', ascending=True)
ic_df_clean = ic_df[(ic_df['Interest Coverage'] > -5) & (ic_df['Interest Coverage'] < 30)]
colors = ['#00CC66' if x > 5 else '#FFAA00' if x > 2 else '#FF4444' for x in ic_df_clean['Interest Coverage']]
fig.add_trace(go.Bar(
    y=ic_df_clean['Name'], x=ic_df_clean['Interest Coverage'],
    orientation='h', marker_color=colors,
    text=ic_df_clean['Interest Coverage'].round(1), textposition='outside',
    showlegend=False
), row=2, col=1)

# Current Ratio
cr_df = lev_df.dropna(subset=['Current Ratio']).sort_values('Current Ratio', ascending=True)
colors = ['#00CC66' if x > 1 else '#FFAA00' if x > 0.5 else '#FF4444' for x in cr_df['Current Ratio']]
fig.add_trace(go.Bar(
    y=cr_df['Name'], x=cr_df['Current Ratio'],
    orientation='h', marker_color=colors,
    text=cr_df['Current Ratio'].round(2), textposition='outside',
    showlegend=False
), row=2, col=2)
fig.add_vline(x=1.0, line_dash="dash", line_color="white", opacity=0.5, row=2, col=2)

fig.update_layout(
    height=1000, width=1400,
    title_text="<b>Financial Health & Leverage Analysis</b>",
    title_font_size=22,
    font=dict(size=9),
)
fig.show()


## 5. Airline-Specific Operational Efficiency

### Key Airline Industry Metrics:
- **Revenue per Employee** — Proxy for labour productivity; airlines with higher values extract more value per worker
- **CAPEX/Revenue** — Fleet investment intensity; too high can indicate aggressive expansion, too low may mean aging fleet
- **Asset Turnover** — How efficiently the airline uses its total assets to generate revenue
- **FCF Yield** — Free cash flow as % of market cap — key for value investors
- **FCF Margin** — Free cash flow as % of revenue — operational cash generation ability

*Note: True CASK (Cost per Available Seat Kilometre) and RASK (Revenue per ASK) require operational data (ASK, RPK) not available via standard financial APIs. We derive proxy metrics from financial statements.*


In [8]:
# ============================================================
# AIRLINE EFFICIENCY METRICS
# ============================================================

eff_cols = ['Name', 'Region', 'Type', 'Revenue/Employee ($K)', 'CAPEX/Revenue (%)', 
            'Asset Turnover', 'FCF Yield (%)', 'FCF Margin (%)', 'Revenue ($B)', 'Employees']
eff_df = df[eff_cols].copy()

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Revenue per Employee ($K)', 'CAPEX / Revenue (%)', 'Asset Turnover',
        'FCF Yield (%)', 'FCF Margin (%)', 'Revenue ($B) vs Employees'
    ),
    vertical_spacing=0.15, horizontal_spacing=0.06
)

# Revenue per Employee
rpe_df = eff_df.dropna(subset=['Revenue/Employee ($K)']).sort_values('Revenue/Employee ($K)', ascending=True)
colors = [TYPE_COLORS.get(t, '#888') for t in rpe_df['Type']]
fig.add_trace(go.Bar(
    y=rpe_df['Name'], x=rpe_df['Revenue/Employee ($K)'],
    orientation='h', marker_color=colors,
    text=rpe_df['Revenue/Employee ($K)'].round(0), textposition='outside',
    showlegend=False
), row=1, col=1)

# CAPEX/Revenue
capex_df = eff_df.dropna(subset=['CAPEX/Revenue (%)']).sort_values('CAPEX/Revenue (%)', ascending=True)
colors = ['#FF4444' if x > 20 else '#FFAA00' if x > 12 else '#00CC66' for x in capex_df['CAPEX/Revenue (%)']]
fig.add_trace(go.Bar(
    y=capex_df['Name'], x=capex_df['CAPEX/Revenue (%)'],
    orientation='h', marker_color=colors,
    text=capex_df['CAPEX/Revenue (%)'].round(1), textposition='outside',
    showlegend=False
), row=1, col=2)

# Asset Turnover
at_df = eff_df.dropna(subset=['Asset Turnover']).sort_values('Asset Turnover', ascending=True)
colors = [region_color(r) for r in at_df['Region']]
fig.add_trace(go.Bar(
    y=at_df['Name'], x=at_df['Asset Turnover'],
    orientation='h', marker_color=colors,
    text=at_df['Asset Turnover'].round(2), textposition='outside',
    showlegend=False
), row=1, col=3)

# FCF Yield
fcfy_df = eff_df.dropna(subset=['FCF Yield (%)']).sort_values('FCF Yield (%)', ascending=True)
colors = ['#00CC66' if x > 5 else '#FFAA00' if x > 0 else '#FF4444' for x in fcfy_df['FCF Yield (%)']]
fig.add_trace(go.Bar(
    y=fcfy_df['Name'], x=fcfy_df['FCF Yield (%)'],
    orientation='h', marker_color=colors,
    text=fcfy_df['FCF Yield (%)'].round(1), textposition='outside',
    showlegend=False
), row=2, col=1)

# FCF Margin
fcfm_df = eff_df.dropna(subset=['FCF Margin (%)']).sort_values('FCF Margin (%)', ascending=True)
colors = ['#00CC66' if x > 5 else '#FFAA00' if x > 0 else '#FF4444' for x in fcfm_df['FCF Margin (%)']]
fig.add_trace(go.Bar(
    y=fcfm_df['Name'], x=fcfm_df['FCF Margin (%)'],
    orientation='h', marker_color=colors,
    text=fcfm_df['FCF Margin (%)'].round(1), textposition='outside',
    showlegend=False
), row=2, col=2)

# Revenue vs Employees scatter
scatter_df = eff_df.dropna(subset=['Revenue ($B)', 'Employees']).copy()
scatter_df['Employees_K'] = scatter_df['Employees'] / 1000
for typ in scatter_df['Type'].unique():
    mask = scatter_df['Type'] == typ
    fig.add_trace(go.Scatter(
        x=scatter_df.loc[mask, 'Employees_K'],
        y=scatter_df.loc[mask, 'Revenue ($B)'],
        mode='markers+text',
        marker=dict(size=12, color=TYPE_COLORS.get(typ, '#888')),
        text=scatter_df.loc[mask, 'Name'],
        textposition='top center',
        textfont=dict(size=7),
        name=typ, showlegend=True
    ), row=2, col=3)

fig.update_xaxes(title_text='Employees (thousands)', row=2, col=3)
fig.update_yaxes(title_text='Revenue ($B)', row=2, col=3)

fig.update_layout(
    height=1000, width=1600,
    title_text="<b>Operational Efficiency & Airline-Specific Metrics</b>",
    title_font_size=22,
    font=dict(size=9),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.show()


## 6. Value vs Quality Matrix

The most actionable chart: plotting **valuation** (EV/EBITDA) against **quality** (Operating Margin). 

Airlines in the **bottom-right quadrant** (high margins, low valuation) are the most attractive value opportunities.


In [9]:
# ============================================================
# VALUE vs QUALITY SCATTER MATRIX
# ============================================================

scatter_data = df[['Name', 'Region', 'Type', 'EV/EBITDA', 'Operating Margin (%)', 
                   'Market Cap ($B)', 'FCF Yield (%)', 'ROE (%)', 'Net Debt/EBITDA']].copy()
scatter_data = scatter_data.dropna(subset=['EV/EBITDA', 'Operating Margin (%)'])
scatter_data = scatter_data[(scatter_data['EV/EBITDA'] > 0) & (scatter_data['EV/EBITDA'] < 30)]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'EV/EBITDA vs Operating Margin (size=Market Cap)',
        'EV/EBITDA vs FCF Yield (size=Market Cap)'
    ),
    horizontal_spacing=0.08
)

# Chart 1: EV/EBITDA vs Operating Margin
for region in scatter_data['Region'].unique():
    mask = scatter_data['Region'] == region
    subset = scatter_data[mask]
    fig.add_trace(go.Scatter(
        x=subset['Operating Margin (%)'],
        y=subset['EV/EBITDA'],
        mode='markers+text',
        marker=dict(
            size=subset['Market Cap ($B)'].clip(2, 50) * 1.5,
            color=region_color(region),
            opacity=0.8,
            line=dict(width=1, color='white')
        ),
        text=subset['Name'],
        textposition='top center',
        textfont=dict(size=8),
        name=region,
        legendgroup=region,
        showlegend=True
    ), row=1, col=1)

# Add quadrant lines
med_ev = scatter_data['EV/EBITDA'].median()
med_om = scatter_data['Operating Margin (%)'].median()
fig.add_hline(y=med_ev, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=1)
fig.add_vline(x=med_om, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=1)
fig.add_annotation(x=med_om + 5, y=med_ev - 2, text="⭐ VALUE ZONE", 
                   font=dict(size=14, color='#00FF88'), showarrow=False, row=1, col=1)

# Chart 2: EV/EBITDA vs FCF Yield
scatter2 = scatter_data.dropna(subset=['FCF Yield (%)'])
for region in scatter2['Region'].unique():
    mask = scatter2['Region'] == region
    subset = scatter2[mask]
    fig.add_trace(go.Scatter(
        x=subset['FCF Yield (%)'],
        y=subset['EV/EBITDA'],
        mode='markers+text',
        marker=dict(
            size=subset['Market Cap ($B)'].clip(2, 50) * 1.5,
            color=region_color(region),
            opacity=0.8,
            line=dict(width=1, color='white')
        ),
        text=subset['Name'],
        textposition='top center',
        textfont=dict(size=8),
        name=region,
        legendgroup=region,
        showlegend=False
    ), row=1, col=2)

fig.update_xaxes(title_text='Operating Margin (%)', row=1, col=1)
fig.update_yaxes(title_text='EV/EBITDA', row=1, col=1)
fig.update_xaxes(title_text='FCF Yield (%)', row=1, col=2)
fig.update_yaxes(title_text='EV/EBITDA', row=1, col=2)

fig.update_layout(
    height=700, width=1500,
    title_text="<b>Value vs Quality Matrix — Find the Undervalued Airlines</b>",
    title_font_size=22,
    font=dict(size=10),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.show()


## 7. Price Performance & Momentum Analysis

Comparing relative price performance across timeframes helps identify airlines with momentum tailwinds or those oversold and potentially ready for mean reversion.


In [10]:
# ============================================================
# PRICE PERFORMANCE HEATMAP
# ============================================================

perf_cols = ['Name', 'Region', 'Type', '1M Return (%)', '3M Return (%)', 
             '6M Return (%)', '1Y Return (%)', 'Dist from 52W High (%)', 'Annualised Vol (%)']
perf_df = df[perf_cols].dropna(subset=['1Y Return (%)']).copy()
perf_df = perf_df.sort_values('1Y Return (%)', ascending=False)

# Heatmap data
heatmap_cols = ['1M Return (%)', '3M Return (%)', '6M Return (%)', '1Y Return (%)', 'Dist from 52W High (%)']
heatmap_data = perf_df[heatmap_cols].values

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=['1M', '3M', '6M', '1Y', 'Dist 52W High'],
    y=perf_df['Name'].values,
    colorscale='RdYlGn',
    zmid=0,
    text=np.round(heatmap_data, 1),
    texttemplate='%{text:.1f}%',
    textfont=dict(size=10),
    hovertemplate='%{y}<br>%{x}: %{z:.1f}%<extra></extra>',
    colorbar=dict(title='Return %')
))

fig.update_layout(
    height=max(600, len(perf_df) * 35),
    width=900,
    title_text="<b>Price Performance Heatmap — Returns Across Timeframes</b>",
    title_font_size=20,
    yaxis=dict(autorange='reversed'),
    font=dict(size=10),
)
fig.show()


In [11]:
# ============================================================
# NORMALISED PRICE PERFORMANCE - 1 YEAR
# ============================================================

fig = go.Figure()

# Get top 10 by market cap for readability
top_airlines = df.nlargest(12, 'Market Cap ($B)').index.tolist()

for ticker in top_airlines:
    if ticker in airline_data:
        hist = airline_data[ticker]['history']
        if hist is not None and len(hist) > 0:
            prices = hist['Close']
            normalised = (prices / prices.iloc[0] - 1) * 100
            fig.add_trace(go.Scatter(
                x=normalised.index,
                y=normalised.values,
                mode='lines',
                name=f"{df.loc[ticker, 'Name']}",
                line=dict(width=2),
                hovertemplate='%{x}<br>%{y:.1f}%<extra></extra>'
            ))

fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.3)

fig.update_layout(
    height=600, width=1400,
    title_text="<b>Normalised 1-Year Price Performance (Top 12 by Market Cap)</b>",
    title_font_size=20,
    xaxis_title="Date",
    yaxis_title="Return (%)",
    legend=dict(font=dict(size=9)),
    hovermode='x unified'
)
fig.show()


## 8. Composite Valuation Score — Finding Undervalued Airlines

### Scoring Methodology

We construct a **multi-factor composite score** that ranks airlines across 12 dimensions. Each metric is converted to a percentile rank (0-100), with direction adjusted so that higher = better.

| Factor | Weight | Direction | Rationale |
|--------|--------|-----------|-----------|
| Forward P/E | 15% | Lower is better | Cheapest on earnings |
| EV/EBITDA | 15% | Lower is better | Core airline valuation metric |
| P/B | 5% | Lower is better | Asset-heavy, book value matters |
| FCF Yield | 12% | Higher is better | Cash generation vs price |
| Operating Margin | 12% | Higher is better | Operational efficiency |
| ROIC | 10% | Higher is better | Capital efficiency |
| Net Debt/EBITDA | 8% | Lower is better | Balance sheet strength |
| Revenue Growth | 5% | Higher is better | Top-line momentum |
| Revenue/Employee | 5% | Higher is better | Labour productivity |
| Interest Coverage | 5% | Higher is better | Debt safety |
| Analyst Upside | 5% | Higher is better | Street consensus |
| 1Y Return | 3% | Lower is better | Contrarian—beaten-down stocks |

Final score ranges from 0-100, where **higher = more attractive**.


In [12]:
# ============================================================
# COMPOSITE VALUATION SCORING ENGINE
# ============================================================

def percentile_rank(series, higher_is_better=True):
    """Convert a series to 0-100 percentile ranks."""
    valid = series.dropna()
    if len(valid) < 3:
        return pd.Series(np.nan, index=series.index)
    ranked = valid.rank(pct=True) * 100
    if not higher_is_better:
        ranked = 100 - ranked
    return ranked.reindex(series.index)

# Define scoring factors
FACTORS = {
    # (column, weight, higher_is_better)
    'Forward P/E':           (0.15, False),
    'EV/EBITDA':             (0.15, False),
    'P/B':                   (0.05, False),
    'FCF Yield (%)':         (0.12, True),
    'Operating Margin (%)':  (0.12, True),
    'ROIC (%)':              (0.10, True),
    'Net Debt/EBITDA':       (0.08, False),
    'Revenue Growth (%)':    (0.05, True),
    'Revenue/Employee ($K)': (0.05, True),
    'Interest Coverage':     (0.05, True),
    'Analyst Upside (%)':    (0.05, True),
    '1Y Return (%)':         (0.03, False),  # Contrarian
}

# Calculate scores
score_df = df.copy()
factor_scores = {}

for factor, (weight, higher_better) in FACTORS.items():
    col_name = f'Score_{factor}'
    # Clean extreme outliers
    clean = score_df[factor].copy()
    if clean.dropna().empty:
        factor_scores[col_name] = pd.Series(np.nan, index=score_df.index)
        continue
    q1, q99 = clean.quantile(0.01), clean.quantile(0.99)
    clean = clean.clip(q1, q99)
    factor_scores[col_name] = percentile_rank(clean, higher_better)

# Build score matrix
for col, scores in factor_scores.items():
    score_df[col] = scores

# Calculate weighted composite score
score_columns = list(factor_scores.keys())
weights = [v[0] for v in FACTORS.values()]

# For each airline, calculate weighted average of available scores
def calc_composite(row):
    scores = []
    ws = []
    for col, w in zip(score_columns, weights):
        if pd.notna(row[col]):
            scores.append(row[col])
            ws.append(w)
    if not ws:
        return np.nan
    # Renormalise weights
    total_w = sum(ws)
    return sum(s * w / total_w for s, w in zip(scores, ws))

score_df['Composite Score'] = score_df.apply(calc_composite, axis=1)
score_df['Rank'] = score_df['Composite Score'].rank(ascending=False, method='min')

# Display results
result_cols = ['Name', 'Region', 'Type', 'Market Cap ($B)', 'Composite Score', 'Rank',
               'Forward P/E', 'EV/EBITDA', 'Operating Margin (%)', 'FCF Yield (%)', 
               'ROIC (%)', 'Net Debt/EBITDA', 'Analyst Upside (%)']
               
results = score_df[result_cols].sort_values('Composite Score', ascending=False)

print("🏆 COMPOSITE VALUATION RANKING — GLOBAL AIRLINES")
print("=" * 130)
print("Higher Composite Score = More Attractive (Undervalued + Quality)")
print("=" * 130)
print(results.to_string(index=True))


🏆 COMPOSITE VALUATION RANKING — GLOBAL AIRLINES
Higher Composite Score = More Attractive (Undervalued + Quality)
                          Name         Region      Type  Market Cap ($B)  Composite Score  Rank  Forward P/E  EV/EBITDA  Operating Margin (%)  FCF Yield (%)  ROIC (%)  Net Debt/EBITDA  Analyst Upside (%)
Ticker                                                                                                                                                                                                     
EZJ.L                  easyJet         Europe       LCC             3.53            74.92  1.00         6.07       2.92                 16.31           5.71     12.44            -0.58               28.30
IAG.L          IAG (BA/Iberia)         Europe       FSC            19.63            69.22  2.00         6.49       4.11                 22.26            NaN     22.35             0.98               16.38
9201.T          Japan Airlines   Asia-Pacific       FSC          1320.0

In [13]:
# ============================================================
# COMPOSITE SCORE VISUALISATION
# ============================================================

ranked = score_df.sort_values('Composite Score', ascending=True).dropna(subset=['Composite Score'])

# Color gradient from red (low) to green (high)
colors = []
for score in ranked['Composite Score']:
    if score >= 70:
        colors.append('#00CC66')  # Strong Buy zone
    elif score >= 55:
        colors.append('#88CC44')  # Buy zone
    elif score >= 40:
        colors.append('#FFAA00')  # Hold zone
    elif score >= 25:
        colors.append('#FF6644')  # Weak
    else:
        colors.append('#FF3333')  # Avoid

fig = go.Figure()

fig.add_trace(go.Bar(
    y=ranked['Name'],
    x=ranked['Composite Score'],
    orientation='h',
    marker_color=colors,
    text=[f"{s:.1f} (#{int(r)})" for s, r in zip(ranked['Composite Score'], ranked['Rank'])],
    textposition='outside',
    textfont=dict(size=11, color='white'),
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Composite Score: %{x:.1f}<br>'
        '<extra></extra>'
    )
))

# Add zone annotations
fig.add_vrect(x0=70, x1=100, fillcolor='green', opacity=0.05, line_width=0)
fig.add_vrect(x0=55, x1=70, fillcolor='lightgreen', opacity=0.05, line_width=0)
fig.add_vrect(x0=0, x1=40, fillcolor='red', opacity=0.05, line_width=0)

fig.add_annotation(x=85, y=len(ranked)-1, text="STRONG BUY", 
                   font=dict(size=12, color='#00FF88'), showarrow=False)
fig.add_annotation(x=62, y=len(ranked)-1, text="BUY", 
                   font=dict(size=12, color='#88CC44'), showarrow=False)
fig.add_annotation(x=47, y=len(ranked)-1, text="HOLD", 
                   font=dict(size=12, color='#FFAA00'), showarrow=False)

fig.update_layout(
    height=max(600, len(ranked) * 38),
    width=1000,
    title_text="<b>🏆 Composite Valuation Score — Global Airlines Ranked</b>",
    title_font_size=22,
    xaxis_title="Composite Score (0-100, Higher = More Attractive)",
    font=dict(size=10),
    xaxis=dict(range=[0, 105])
)
fig.show()


## 9. Factor Breakdown — Top 5 Airlines

Radar charts showing how the top-ranked airlines score across each individual factor.


In [14]:
# ============================================================
# RADAR CHARTS - TOP 5 AIRLINES
# ============================================================

top_n = min(6, len(score_df.dropna(subset=['Composite Score'])))
top_airlines_scored = score_df.nlargest(top_n, 'Composite Score')

# Factor names for radar
radar_factors = list(FACTORS.keys())
radar_labels = [f.replace('(%)', '').replace('($K)', '').strip() for f in radar_factors]

fig = go.Figure()

radar_colors = ['#00D4AA', '#FF6B6B', '#4ECDC4', '#FFE66D', '#FF8C42', '#AB63FA']

for i, (ticker, row) in enumerate(top_airlines_scored.iterrows()):
    values = []
    for factor in radar_factors:
        col = f'Score_{factor}'
        val = row.get(col, np.nan)
        values.append(val if pd.notna(val) else 0)
    
    # Close the radar
    values_closed = values + [values[0]]
    labels_closed = radar_labels + [radar_labels[0]]
    
    fig.add_trace(go.Scatterpolar(
        r=values_closed,
        theta=labels_closed,
        fill='toself',
        name=f"#{int(row['Rank'])} {row['Name']}",
        line_color=radar_colors[i % len(radar_colors)],
        opacity=0.6,
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 100]),
    ),
    height=700, width=900,
    title_text=f"<b>Factor Breakdown — Top {top_n} Ranked Airlines</b>",
    title_font_size=20,
    font=dict(size=10),
    legend=dict(font=dict(size=11))
)
fig.show()


## 10. Regional & Carrier Type Analysis

How do airlines compare when grouped by **region** and **carrier type** (FSC vs LCC vs ULCC)?


In [15]:
# ============================================================
# REGIONAL AND TYPE COMPARISON
# ============================================================

# Group by region
region_metrics = ['Operating Margin (%)', 'EBITDA Margin (%)', 'Net Margin (%)', 
                  'EV/EBITDA', 'Forward P/E', 'FCF Yield (%)', 'ROE (%)', 'Net Debt/EBITDA']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Operating Margin by Region', 'EV/EBITDA by Region',
        'Operating Margin by Carrier Type', 'EV/EBITDA by Carrier Type'
    ),
    vertical_spacing=0.15
)

# Region box plots
for col_name, row, col_pos in [('Operating Margin (%)', 1, 1), ('EV/EBITDA', 1, 2)]:
    for region in sorted(df['Region'].unique()):
        mask = df['Region'] == region
        vals = df.loc[mask, col_name].dropna()
        if len(vals) > 0:
            fig.add_trace(go.Box(
                y=vals, name=region,
                marker_color=region_color(region),
                boxmean=True,
                showlegend=(col_pos == 1),
                legendgroup=region
            ), row=row, col=col_pos)

# Type box plots
for col_name, row, col_pos in [('Operating Margin (%)', 2, 1), ('EV/EBITDA', 2, 2)]:
    for ctype in sorted(df['Type'].unique()):
        mask = df['Type'] == ctype
        vals = df.loc[mask, col_name].dropna()
        if len(vals) > 0:
            fig.add_trace(go.Box(
                y=vals, name=ctype,
                marker_color=TYPE_COLORS.get(ctype, '#888'),
                boxmean=True,
                showlegend=False
            ), row=row, col=col_pos)

fig.update_layout(
    height=900, width=1300,
    title_text="<b>Regional & Carrier Type Comparison</b>",
    title_font_size=22,
    font=dict(size=10),
)
fig.show()

# Summary statistics table
print("\n📊 MEDIAN METRICS BY REGION:")
print("=" * 100)
region_summary = df.groupby('Region')[region_metrics].median().round(2)
print(region_summary.to_string())

print("\n📊 MEDIAN METRICS BY CARRIER TYPE:")
print("=" * 100)
type_summary = df.groupby('Type')[region_metrics].median().round(2)
print(type_summary.to_string())



📊 MEDIAN METRICS BY REGION:
               Operating Margin (%)  EBITDA Margin (%)  Net Margin (%)  EV/EBITDA  Forward P/E  FCF Yield (%)  ROE (%)  Net Debt/EBITDA
Region                                                                                                                                 
Asia-Pacific                  10.91              17.15            6.49       5.99        12.25           7.57    13.85             1.42
Europe                         6.73              11.35            5.33       4.57         6.80           5.20    20.85             2.48
Europe/ME                      0.00                NaN            5.16        NaN        13.08          93.46     6.65            -0.13
Latin America                 21.77              32.73           18.57       5.78         7.09           1.14    26.09             0.85
North America                  5.09              10.00            1.57       8.00         8.03           0.26    10.72             2.68

📊 MEDIAN METRICS B

## 11. Dividend & Shareholder Return Analysis

For income-oriented investors, dividend yield and payout ratios matter. Many airlines suspended dividends during COVID — those that have resumed signal financial confidence.


In [16]:
# ============================================================
# DIVIDEND ANALYSIS
# ============================================================

div_df = df[['Name', 'Region', 'Type', 'Dividend Yield (%)', 'Payout Ratio (%)', 
             'FCF Yield (%)', 'Market Cap ($B)']].copy()
div_df = div_df[div_df['Dividend Yield (%)'] > 0].sort_values('Dividend Yield (%)', ascending=False)

if len(div_df) > 0:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Dividend Yield (%)', 'Dividend Yield vs FCF Yield'),
        horizontal_spacing=0.1
    )
    
    colors = [region_color(r) for r in div_df['Region']]
    fig.add_trace(go.Bar(
        x=div_df['Name'], y=div_df['Dividend Yield (%)'],
        marker_color=colors,
        text=div_df['Dividend Yield (%)'].round(2),
        textposition='outside',
        showlegend=False
    ), row=1, col=1)
    
    # Dividend vs FCF Yield scatter
    scatter_div = div_df.dropna(subset=['FCF Yield (%)'])
    for region in scatter_div['Region'].unique():
        mask = scatter_div['Region'] == region
        subset = scatter_div[mask]
        fig.add_trace(go.Scatter(
            x=subset['FCF Yield (%)'],
            y=subset['Dividend Yield (%)'],
            mode='markers+text',
            marker=dict(size=15, color=region_color(region)),
            text=subset['Name'],
            textposition='top center',
            textfont=dict(size=8),
            name=region,
        ), row=1, col=2)
    
    # 45-degree line (dividend = FCF yield means 100% payout)
    max_val = max(scatter_div['FCF Yield (%)'].max(), scatter_div['Dividend Yield (%)'].max()) * 1.1
    fig.add_trace(go.Scatter(
        x=[0, max_val], y=[0, max_val],
        mode='lines', line=dict(dash='dash', color='gray'),
        showlegend=False, name='100% Payout'
    ), row=1, col=2)
    
    fig.update_xaxes(title_text='FCF Yield (%)', row=1, col=2)
    fig.update_yaxes(title_text='Dividend Yield (%)', row=1, col=2)
    
    fig.update_layout(
        height=500, width=1300,
        title_text="<b>Dividend & Shareholder Returns</b>",
        title_font_size=22,
        font=dict(size=10),
    )
    fig.show()
    
    print("\n📊 DIVIDEND-PAYING AIRLINES:")
    print(div_df[['Name', 'Dividend Yield (%)', 'Payout Ratio (%)', 'FCF Yield (%)']].to_string(index=True))
else:
    print("No airlines currently paying dividends in our universe.")



📊 DIVIDEND-PAYING AIRLINES:
                          Name  Dividend Yield (%)  Payout Ratio (%)  FCF Yield (%)
Ticker                                                                             
0293.HK         Cathay Pacific              542.00             48.94          16.82
C6L.SI      Singapore Airlines              507.00             56.26           7.95
CPA              Copa Holdings              481.00             39.56           1.14
AIR.NZ         Air New Zealand              439.00             74.32          21.59
LHA.DE         Lufthansa Group              330.00             22.06         -11.14
TKC           Turkish Airlines              315.00             73.17          93.46
QAN.AX          Qantas Airways              314.00             15.87           1.86
9201.T          Japan Airlines              301.00             32.74           7.19
EZJ.L                  easyJet              279.00             18.70           5.71
IAG.L          IAG (BA/Iberia)              213

## 12. Financial Statements — Revenue & Profitability Trends

Tracking multi-year trends in revenue, EBITDA, and margins for the largest airlines.


In [17]:
# ============================================================
# MULTI-YEAR FINANCIAL TRENDS
# ============================================================

# Select top airlines by market cap for trend analysis
top_tickers = df.nlargest(8, 'Market Cap ($B)').index.tolist()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Revenue Trend ($B)', 'EBITDA Trend ($B)',
        'Operating Income Trend ($B)', 'Free Cash Flow Trend ($B)'
    ),
    vertical_spacing=0.12, horizontal_spacing=0.08
)

metrics_to_plot = [
    ('Total Revenue', 1, 1),
    ('EBITDA', 1, 2),
    ('Operating Income', 2, 1),
]

line_colors = px.colors.qualitative.Set2

for idx, ticker in enumerate(top_tickers):
    if ticker not in airline_data:
        continue
    data = airline_data[ticker]
    name = data['meta']['name']
    color = line_colors[idx % len(line_colors)]
    
    # Revenue, EBITDA, Operating Income from income statement
    inc = data['income']
    cf = data['cashflow']
    
    for metric, row, col in metrics_to_plot:
        if inc is not None and metric in inc.index:
            vals = inc.loc[metric].dropna()
            years = [d.strftime('%Y') for d in vals.index]
            fig.add_trace(go.Scatter(
                x=years, y=vals.values / 1e9,
                mode='lines+markers',
                name=name if (row == 1 and col == 1) else None,
                showlegend=(row == 1 and col == 1),
                legendgroup=name,
                line=dict(color=color, width=2),
                marker=dict(size=6),
                hovertemplate=f'{name}<br>%{{x}}: $%{{y:.1f}}B<extra></extra>'
            ), row=row, col=col)
    
    # FCF from cash flow
    if cf is not None and 'Free Cash Flow' in cf.index:
        vals = cf.loc['Free Cash Flow'].dropna()
        years = [d.strftime('%Y') for d in vals.index]
        fig.add_trace(go.Scatter(
            x=years, y=vals.values / 1e9,
            mode='lines+markers',
            name=None, showlegend=False,
            legendgroup=name,
            line=dict(color=color, width=2),
            marker=dict(size=6),
        ), row=2, col=2)

for row in [1, 2]:
    for col in [1, 2]:
        fig.update_yaxes(title_text='$B', row=row, col=col)

fig.update_layout(
    height=800, width=1400,
    title_text="<b>Multi-Year Financial Trends — Top Airlines</b>",
    title_font_size=22,
    font=dict(size=9),
    legend=dict(font=dict(size=9))
)
fig.show()


In [18]:
# ============================================================
# BALANCE SHEET TRENDS - Debt & Equity
# ============================================================

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Total Debt Trend ($B)', 'Total Equity Trend ($B)'),
    horizontal_spacing=0.08
)

for idx, ticker in enumerate(top_tickers):
    if ticker not in airline_data:
        continue
    data = airline_data[ticker]
    name = data['meta']['name']
    color = line_colors[idx % len(line_colors)]
    bs = data['balance']
    
    if bs is None:
        continue
    
    for metric, col in [('Total Debt', 1), ('Stockholders Equity', 2)]:
        if metric in bs.index:
            vals = bs.loc[metric].dropna()
            years = [d.strftime('%Y') for d in vals.index]
            fig.add_trace(go.Scatter(
                x=years, y=vals.values / 1e9,
                mode='lines+markers',
                name=name if col == 1 else None,
                showlegend=(col == 1),
                legendgroup=name,
                line=dict(color=color, width=2),
                marker=dict(size=6),
            ), row=1, col=col)

fig.update_yaxes(title_text='$B', row=1, col=1)
fig.update_yaxes(title_text='$B', row=1, col=2)

fig.update_layout(
    height=500, width=1400,
    title_text="<b>Balance Sheet Trends — Debt & Equity</b>",
    title_font_size=22,
    font=dict(size=9),
)
fig.show()


## 13. Correlation Matrix — Metric Relationships

Understanding how different metrics correlate helps validate our scoring approach and identify key drivers of airline valuation.


In [19]:
# ============================================================
# CORRELATION ANALYSIS
# ============================================================

corr_cols = ['Trailing P/E', 'EV/EBITDA', 'P/B', 'Operating Margin (%)', 
             'Net Margin (%)', 'ROE (%)', 'ROIC (%)', 'Debt/Equity', 
             'Net Debt/EBITDA', 'FCF Yield (%)', 'Revenue Growth (%)',
             'Asset Turnover', '1Y Return (%)', 'Beta', 'Composite Score']

corr_data = score_df[corr_cols].dropna(thresh=len(corr_cols)//2)
corr_matrix = corr_data.corr()

# Clean labels
short_labels = [c.replace('(%)', '').replace('($K)', '').strip() for c in corr_cols]

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=short_labels,
    y=short_labels,
    colorscale='RdBu_r',
    zmid=0,
    text=np.round(corr_matrix.values, 2),
    texttemplate='%{text:.2f}',
    textfont=dict(size=8),
    colorbar=dict(title='Correlation')
))

fig.update_layout(
    height=800, width=900,
    title_text="<b>Metric Correlation Matrix</b>",
    title_font_size=20,
    font=dict(size=9),
    xaxis=dict(tickangle=45),
)
fig.show()


## 14. Risk-Return Analysis

Plotting **return vs volatility** reveals which airlines offer the best risk-adjusted returns. The **Sharpe-like ratio** (return/volatility) helps identify efficient performers.


In [20]:
# ============================================================
# RISK-RETURN SCATTER
# ============================================================

rr_df = df[['Name', 'Region', 'Type', '1Y Return (%)', 'Annualised Vol (%)', 
            'Market Cap ($B)', 'Beta']].dropna(subset=['1Y Return (%)', 'Annualised Vol (%)']).copy()

# Calculate simple Sharpe-like ratio (return / vol)
rr_df['Return/Risk'] = rr_df['1Y Return (%)'] / rr_df['Annualised Vol (%)']

fig = go.Figure()

for region in rr_df['Region'].unique():
    mask = rr_df['Region'] == region
    subset = rr_df[mask]
    fig.add_trace(go.Scatter(
        x=subset['Annualised Vol (%)'],
        y=subset['1Y Return (%)'],
        mode='markers+text',
        marker=dict(
            size=subset['Market Cap ($B)'].clip(3, 50) * 1.5,
            color=region_color(region),
            opacity=0.8,
            line=dict(width=1, color='white')
        ),
        text=subset['Name'],
        textposition='top center',
        textfont=dict(size=8),
        name=region,
    ))

# Add risk-free rate reference
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

fig.update_layout(
    height=600, width=1100,
    title_text="<b>Risk-Return Analysis — 1 Year (Size = Market Cap)</b>",
    title_font_size=20,
    xaxis_title="Annualised Volatility (%)",
    yaxis_title="1-Year Return (%)",
    font=dict(size=10),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.show()


## 15. Master Summary Table

Complete metrics for all airlines, sorted by composite valuation score.


In [21]:
# ============================================================
# MASTER SUMMARY TABLE
# ============================================================

summary_cols = [
    'Name', 'Region', 'Type', 'Market Cap ($B)', 'Revenue ($B)',
    'Composite Score', 'Rank',
    # Valuation
    'Trailing P/E', 'Forward P/E', 'EV/EBITDA', 'P/B', 'P/S', 'EV/Revenue',
    # Profitability
    'Operating Margin (%)', 'EBITDA Margin (%)', 'Net Margin (%)', 'ROE (%)', 'ROA (%)', 'ROIC (%)',
    # Leverage
    'Debt/Equity', 'Net Debt/EBITDA', 'Interest Coverage', 'Current Ratio',
    # Cash Flow
    'FCF Yield (%)', 'FCF Margin (%)',
    # Efficiency
    'Revenue/Employee ($K)', 'CAPEX/Revenue (%)', 'Asset Turnover',
    # Growth
    'Revenue Growth (%)', 'Earnings Growth (%)',
    # Performance
    '1Y Return (%)', 'Annualised Vol (%)', 'Beta',
    # Dividend
    'Dividend Yield (%)',
    # Analyst
    'Analyst Upside (%)', 'Analyst Recommendation'
]

available_cols = [c for c in summary_cols if c in score_df.columns]
master = score_df[available_cols].sort_values('Composite Score', ascending=False)

# Style function for Jupyter display
print("\n" + "=" * 150)
print("📋 MASTER SUMMARY TABLE — ALL AIRLINES RANKED BY COMPOSITE SCORE")
print("=" * 150)
print(master.round(2).to_string(index=True))



📋 MASTER SUMMARY TABLE — ALL AIRLINES RANKED BY COMPOSITE SCORE
                          Name         Region      Type  Market Cap ($B)  Revenue ($B)  Composite Score  Rank  Trailing P/E  Forward P/E  EV/EBITDA    P/B  P/S  EV/Revenue  Operating Margin (%)  EBITDA Margin (%)  Net Margin (%)  ROE (%)  ROA (%)  ROIC (%)  Debt/Equity  Net Debt/EBITDA  Interest Coverage  Current Ratio  FCF Yield (%)  FCF Margin (%)  Revenue/Employee ($K)  CAPEX/Revenue (%)  Asset Turnover  Revenue Growth (%)  Earnings Growth (%)  1Y Return (%)  Annualised Vol (%)  Beta  Dividend Yield (%)  Analyst Upside (%) Analyst Recommendation
Ticker                                                                                                                                                                                                                                                                                                                                                                                      

## 16. Investment Conclusions & Key Findings

### Interpretation Guide

**Composite Score Zones:**
- **≥ 70:** Strong Buy — Significantly undervalued with quality characteristics
- **55-70:** Buy — Attractively valued with solid fundamentals  
- **40-55:** Hold — Fairly valued, situation-dependent
- **< 40:** Avoid/Sell — Overvalued or deteriorating fundamentals

### Key Caveats for Airline Investing:
1. **Cyclicality** — Airlines are highly cyclical; current metrics may not reflect mid-cycle earnings
2. **Fuel Exposure** — Jet fuel is ~25-35% of costs; hedging policies vary significantly
3. **Currency Risk** — International airlines have complex currency exposures
4. **Lease Accounting** — IFRS 16/ASC 842 changes make debt comparisons tricky
5. **Load Factor** — Not available via financial APIs but critical; check investor presentations
6. **CASK/RASK** — True cost/revenue per available seat km requires operational data from annual reports
7. **Fleet Age** — Younger fleets have lower maintenance but higher depreciation
8. **Route Mix** — Long-haul vs short-haul economics differ dramatically
9. **Government Ownership** — Some airlines (e.g., Turkish, Air China) have state involvement affecting governance

### Suggested Next Steps:
- For top-ranked airlines, read latest earnings call transcripts
- Check fleet order books (Airbus/Boeing backlogs) for growth capex commitments
- Monitor fuel hedge positions in annual reports
- Verify load factors and yield data from IATA statistics
- Consider macro factors: GDP growth in home markets, tourism recovery trends, open skies agreements

---
*This analysis was generated using publicly available financial data via Yahoo Finance. Past performance does not guarantee future results. This is not financial advice.*


In [22]:
# ============================================================
# FINAL SUMMARY - TOP PICKS
# ============================================================

print("🏆 TOP 5 MOST ATTRACTIVE AIRLINE STOCKS (by Composite Score)")
print("=" * 100)

top5 = score_df.nlargest(5, 'Composite Score')
for i, (ticker, row) in enumerate(top5.iterrows(), 1):
    print(f"\n{'─' * 80}")
    print(f"  #{i}  {row['Name']} ({ticker})")
    print(f"{'─' * 80}")
    print(f"  Composite Score: {row['Composite Score']:.1f}/100")
    print(f"  Region: {row['Region']}  |  Type: {row['Type']}  |  Market Cap: ${row['Market Cap ($B)']:.1f}B")
    print(f"  Forward P/E: {row.get('Forward P/E', 'N/A'):.1f}x  |  EV/EBITDA: {row.get('EV/EBITDA', 'N/A'):.1f}x")
    print(f"  Operating Margin: {row.get('Operating Margin (%)', 'N/A'):.1f}%  |  ROIC: {row.get('ROIC (%)', 'N/A'):.1f}%")
    print(f"  FCF Yield: {row.get('FCF Yield (%)', 'N/A'):.1f}%  |  Net Debt/EBITDA: {row.get('Net Debt/EBITDA', 'N/A'):.1f}x")
    print(f"  1Y Return: {row.get('1Y Return (%)', 'N/A'):.1f}%  |  Analyst Upside: {row.get('Analyst Upside (%)', 'N/A'):.1f}%")

print(f"\n\n{'=' * 100}")
print("⚠️  DISCLAIMER: This analysis is for educational purposes only.")
print("    Always conduct your own research before making investment decisions.")
print(f"{'=' * 100}")


🏆 TOP 5 MOST ATTRACTIVE AIRLINE STOCKS (by Composite Score)

────────────────────────────────────────────────────────────────────────────────
  #1  easyJet (EZJ.L)
────────────────────────────────────────────────────────────────────────────────
  Composite Score: 74.9/100
  Region: Europe  |  Type: LCC  |  Market Cap: $3.5B
  Forward P/E: 6.1x  |  EV/EBITDA: 2.9x
  Operating Margin: 16.3%  |  ROIC: 12.4%
  FCF Yield: 5.7%  |  Net Debt/EBITDA: -0.6x
  1Y Return: -4.5%  |  Analyst Upside: 28.3%

────────────────────────────────────────────────────────────────────────────────
  #2  IAG (BA/Iberia) (IAG.L)
────────────────────────────────────────────────────────────────────────────────
  Composite Score: 69.2/100
  Region: Europe  |  Type: FSC  |  Market Cap: $19.6B
  Forward P/E: 6.5x  |  EV/EBITDA: 4.1x
  Operating Margin: 22.3%  |  ROIC: 22.3%
  FCF Yield: nan%  |  Net Debt/EBITDA: 1.0x
  1Y Return: 31.8%  |  Analyst Upside: 16.4%

───────────────────────────────────────────────────────

In [23]:
# ============================================================
# EXPORT RESULTS TO CSV
# ============================================================

# Export master table
master.to_csv('airline_valuation_analysis.csv')
print("✅ Results exported to 'airline_valuation_analysis.csv'")
print(f"\n📊 Analysis complete! {len(df)} airlines analysed across {len(df.columns)} metrics.")


✅ Results exported to 'airline_valuation_analysis.csv'

📊 Analysis complete! 24 airlines analysed across 53 metrics.


In [24]:
!pip install nbconvert -q

In [25]:
import glob

# Find ALL .ipynb files anywhere accessible
for path in glob.glob('/kaggle/**/*.ipynb', recursive=True):
    print(path)

for path in glob.glob('/tmp/**/*.ipynb', recursive=True):
    print(path)

/kaggle/src/script.ipynb
/kaggle/working/__notebook__.ipynb


In [26]:
# ============================================================
# EXPORT NOTEBOOK TO INTERACTIVE HTML
# ============================================================
import subprocess

# Convert notebook to HTML (preserves interactive Plotly charts)
subprocess.run([
    'jupyter', 'nbconvert',
    '--to', 'html',
    '--no-input',
    '/kaggle/working/__notebook__.ipynb',
    '--output', '/kaggle/working/airline_dashboard.html'
], check=True)

print("✅ Exported to Global_Airline_Stock_Valuation_Dashboard.html")

/usr/local/lib/python3.12/dist-packages/mistune.py:435: SyntaxWarning: invalid escape sequence '\|'
  cells[i][c] = re.sub('\\\\\|', '|', cell)
/usr/local/lib/python3.12/dist-packages/nbconvert/filters/filter_links.py:36: SyntaxWarning: invalid escape sequence '\_'
  text = re.sub(r'_', '\_', text) # Escape underscores in display text
[NbConvertApp] Converting notebook /kaggle/working/__notebook__.ipynb to html
[NbConvertApp] Writing 997709 bytes to /kaggle/working/airline_dashboard.html


✅ Exported to Global_Airline_Stock_Valuation_Dashboard.html
